<a href="https://colab.research.google.com/github/maru415/ECON-499/blob/main/foottraffic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
from getpass import getpass

API_URL = (
    "https://api.deweydata.io/api/v1/external/data/"
    "prj_qz3n4nfy__cdst_ksf3bqzjycrqkdpr"
)

api_key = getpass("Enter your Dewey API Key: ").strip()

headers = {"X-API-KEY": api_key}

response = requests.get(
    API_URL,
    headers=headers,
    params={"page": 1},
    timeout=60
)

print("HTTP Status:", response.status_code)

response.raise_for_status()

data = response.json()

print("Connection successful!")
print("Total files:", data.get("total_files"))
print("Total pages:", data.get("total_pages"))

Enter your Dewey API Key: ··········
HTTP Status: 200
Connection successful!
Total files: 94
Total pages: 2


In [2]:
all_files = []

total_pages = int(data["total_pages"])

for page in range(1, total_pages + 1):

    r = requests.get(
        API_URL,
        headers=headers,
        params={"page": page},
        timeout=60
    )

    r.raise_for_status()

    page_data = r.json()

    all_files.extend(
        page_data["download_links"]
    )

# Remove duplicate files
all_files = list({
    f["external_id"]: f
    for f in all_files
}.values())

# Select 2024 data
files_2024 = [
    f for f in all_files
    if str(f.get("partition_key", "")).startswith("2024-")
]

files_2024.sort(
    key=lambda f: f["partition_key"]
)

print("Total available files:", len(all_files))
print("2024 files:", len(files_2024))

for f in files_2024:
    print(
        f["partition_key"],
        round(f["file_size_bytes"] / 1e6, 2),
        "MB"
    )

print(
    "\nTotal 2024 size:",
    round(
        sum(f["file_size_bytes"] for f in files_2024) / 1e9,
        2
    ),
    "GB"
)

Total available files: 94
2024 files: 12
2024-01-01 277.4 MB
2024-02-01 297.82 MB
2024-03-01 361.18 MB
2024-04-01 383.26 MB
2024-05-01 390.99 MB
2024-06-01 394.71 MB
2024-07-01 411.19 MB
2024-08-01 375.68 MB
2024-09-01 380.26 MB
2024-10-01 404.89 MB
2024-11-01 399.58 MB
2024-12-01 401.93 MB

Total 2024 size: 4.48 GB


In [3]:
import requests
from pathlib import Path

january = next(
    f for f in files_2024
    if f["partition_key"] == "2024-01-01"
)

output = Path("/content/dewey_january_2024.parquet")

if not output.exists():
    with requests.get(
        january["link"],
        stream=True,
        timeout=(30, 180)
    ) as r:
        r.raise_for_status()

        with open(str(output) + ".part", "wb") as f:
            for chunk in r.iter_content(4 * 1024 * 1024):
                if chunk:
                    f.write(chunk)

    Path(str(output) + ".part").rename(output)

print("Download complete!")
print("File size:", round(output.stat().st_size / 1e6, 2), "MB")

Download complete!
File size: 277.4 MB


In [4]:
%pip -q install pyarrow

In [5]:
import pyarrow.parquet as pq

pf = pq.ParquetFile("/content/dewey_january_2024.parquet")

print("Total rows:", pf.metadata.num_rows)
print("Columns:")

for column in pf.schema_arrow.names:
    print(column)

Total rows: 7432
Columns:
AFTERNOON_TEA_DEVICE_HOME_AREAS
AREA
AREA_TYPE
BREAKFAST_DEVICE_HOME_AREAS
DATE_RANGE_END
DATE_RANGE_START
DAY_COUNTS
DEVICE_COUNTS
DEVICE_DAYTIME_AREAS
DEVICE_HOME_AREAS
DEVICE_TYPE
DINNER_DEVICE_HOME_AREAS
DISTANCE_FROM_HOME
DISTANCE_FROM_PRIMARY_DAYTIME_LOCATION
ISO_COUNTRY_CODE
LUNCH_DEVICE_HOME_AREAS
MEDIAN_DWELL
MONTH
NIGHTLIFE_DEVICE_HOME_AREAS
ORIGIN_AREA_TYPE
POPULARITY_BY_EACH_HOUR
POPULARITY_BY_HOUR_FRIDAY
POPULARITY_BY_HOUR_MONDAY
POPULARITY_BY_HOUR_SATURDAY
POPULARITY_BY_HOUR_SUNDAY
POPULARITY_BY_HOUR_THURSDAY
POPULARITY_BY_HOUR_TUESDAY
POPULARITY_BY_HOUR_WEDNESDAY
REGION
STOPS_BY_DAY
STOPS_BY_EACH_HOUR
STOP_COUNTS
TOP_SAME_DAY_BRAND
TOP_SAME_MONTH_BRAND
WEEKDAY_DEVICE_HOME_AREAS
WEEKEND_DEVICE_HOME_AREAS
WORK_BEHAVIOR_DEVICE_HOME_AREAS
WORK_HOURS_DEVICE_HOME_AREAS
YEAR


In [6]:
import pandas as pd

mall_cbgs = {
    "Lenox Square": "131210096061",
    "Phipps Plaza": "131210100062",
    "Ponce City Market": "131210013021"
}

wanted = [
    "AREA",
    "AREA_TYPE",
    "ORIGIN_AREA_TYPE",
    "DEVICE_COUNTS",
    "STOP_COUNTS"
]

results = []

for batch in pf.iter_batches(
    batch_size=20000,
    columns=wanted
):
    df = batch.to_pandas()

    mask = (
        df["AREA"].astype(str).isin(mall_cbgs.values())
        & df["AREA_TYPE"].eq("Census Block Group")
        & df["ORIGIN_AREA_TYPE"].eq("Census Block Group")
    )

    if mask.any():
        results.append(df.loc[mask])

if results:
    matched = pd.concat(results, ignore_index=True)
    matched["Mall"] = matched["AREA"].astype(str).map(
        {v: k for k, v in mall_cbgs.items()}
    )
    display(matched)
else:
    print("No matching records.")

,AREA,AREA_TYPE,ORIGIN_AREA_TYPE,DEVICE_COUNTS,STOP_COUNTS,Mall
0,131210096061,Census Block Group,Census Block Group,330235,1083628,Lenox Square
1,131210100062,Census Block Group,Census Block Group,222577,798883,Phipps Plaza
2,131210013021,Census Block Group,Census Block Group,180812,384579,Ponce City Market


In [15]:
import pandas as pd
import pyarrow.parquet as pq

mall_cbgs = {
    "Lenox Square": "131210096061",
    "Phipps Plaza": "131210100062",
    "Ponce City Market": "131210013021",
    "Greenbriar Mall": "131210077052"
}

target_cbgs = set(mall_cbgs.values())

pf = pq.ParquetFile("/content/dewey_january_2024.parquet")

columns = [
    "AREA",
    "AREA_TYPE",
    "ORIGIN_AREA_TYPE",
    "DEVICE_COUNTS",
    "STOP_COUNTS"
]

results = []

for batch in pf.iter_batches(columns=columns, batch_size=10000):
    df = batch.to_pandas()

    selected = df[
        df["AREA"].astype(str).isin(target_cbgs)
        & df["AREA_TYPE"].eq("Census Block Group")
        & df["ORIGIN_AREA_TYPE"].eq("Census Block Group")
    ]

    if not selected.empty:
        results.append(selected)

if results:
    january_malls = pd.concat(results, ignore_index=True)
    january_malls["Mall"] = january_malls["AREA"].astype(str).map(
        {cbg: mall for mall, cbg in mall_cbgs.items()}
    )

    display(
        january_malls[
            ["Mall", "AREA", "DEVICE_COUNTS", "STOP_COUNTS"]
        ].sort_values("Mall")
    )
else:
    print("No matching destination CBGs found.")

found_cbgs = set(
    january_malls["AREA"].astype(str)
) if results else set()

print("\nDESTINATION VERIFICATION")
print("-" * 45)

for mall, cbg in mall_cbgs.items():
    status = "FOUND" if cbg in found_cbgs else "NOT FOUND"
    print(f"{mall}: {status}")

,Mall,AREA,DEVICE_COUNTS,STOP_COUNTS
0,Greenbriar Mall,131210077052,64075,138579
1,Lenox Square,131210096061,330235,1083628
2,Phipps Plaza,131210100062,222577,798883
3,Ponce City Market,131210013021,180812,384579



DESTINATION VERIFICATION
---------------------------------------------
Lenox Square: FOUND
Phipps Plaza: FOUND
Ponce City Market: FOUND
Greenbriar Mall: FOUND


In [16]:
import json
from collections.abc import Mapping
import pandas as pd

cbg_to_mall = {
    cbg: mall for mall, cbg in mall_cbgs.items()
}

def parse_home_areas(value):
    if value is None:
        return {}

    if isinstance(value, Mapping):
        return dict(value)

    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, dict):
                return parsed
        except (json.JSONDecodeError, TypeError):
            return {}

    return {}

origin_rows = []
destination_rows = []

columns = [
    "AREA",
    "AREA_TYPE",
    "ORIGIN_AREA_TYPE",
    "DEVICE_COUNTS",
    "STOP_COUNTS",
    "DEVICE_HOME_AREAS"
]

for batch in pf.iter_batches(
    columns=columns,
    batch_size=10000
):
    df = batch.to_pandas()

    selected = df[
        df["AREA"].astype(str).isin(target_cbgs)
        & df["AREA_TYPE"].eq("Census Block Group")
        & df["ORIGIN_AREA_TYPE"].eq("Census Block Group")
    ]

    for _, row in selected.iterrows():
        destination_cbg = str(row["AREA"])
        mall = cbg_to_mall[destination_cbg]

        home_areas = parse_home_areas(
            row["DEVICE_HOME_AREAS"]
        )

        destination_rows.append({
            "Mall": mall,
            "Destination_CBG": destination_cbg,
            "Year": 2024,
            "Month": 1,
            "Device_Counts": row["DEVICE_COUNTS"],
            "Stop_Counts": row["STOP_COUNTS"],
            "Reported_Origins": len(home_areas),
            "Reported_Origin_Counts": sum(
                float(count)
                for count in home_areas.values()
                if count is not None
            )
        })

        for origin_cbg, count in home_areas.items():
            if count is None:
                continue

            origin_rows.append({
                "Mall": mall,
                "Destination_CBG": destination_cbg,
                "Origin_CBG": str(origin_cbg),
                "Year": 2024,
                "Month": 1,
                "Origin_Device_Count": float(count)
            })

january_origin_flows = pd.DataFrame(origin_rows)
january_destination_summary = pd.DataFrame(
    destination_rows
)

print("JANUARY 2024 DESTINATION SUMMARY")
display(
    january_destination_summary.sort_values("Mall")
)

print("\nJANUARY 2024 ORIGIN FLOW SUMMARY")
print("Total origin-destination rows:", len(january_origin_flows))
print(
    "Unique origin CBGs:",
    january_origin_flows["Origin_CBG"].nunique()
)

print("\nORIGIN COUNTS BY MALL")
display(
    january_origin_flows.groupby("Mall").agg(
        Reported_Origins=("Origin_CBG", "nunique"),
        Origin_Device_Counts=("Origin_Device_Count", "sum")
    ).reset_index()
)

JANUARY 2024 DESTINATION SUMMARY


,Mall,Destination_CBG,Year,Month,Device_Counts,Stop_Counts,Reported_Origins,Reported_Origin_Counts
0,Greenbriar Mall,131210077052,2024,1,64075,138579,1803,47125.0
1,Lenox Square,131210096061,2024,1,330235,1083628,2000,151494.0
2,Phipps Plaza,131210100062,2024,1,222577,798883,2000,118022.0
3,Ponce City Market,131210013021,2024,1,180812,384579,2000,98028.0



JANUARY 2024 ORIGIN FLOW SUMMARY
Total origin-destination rows: 7803
Unique origin CBGs: 4422

ORIGIN COUNTS BY MALL


,Mall,Reported_Origins,Origin_Device_Counts
0,Greenbriar Mall,1803,47125.0
1,Lenox Square,2000,151494.0
2,Phipps Plaza,2000,118022.0
3,Ponce City Market,2000,98028.0


In [20]:
import os
import gc
import tempfile
import requests
import pandas as pd

def get_download_url(file_record):
    for key in ["link", "download_url", "url"]:
        value = file_record.get(key)
        if isinstance(value, str) and value.startswith("http"):
            return value
    return None


for file_record in sorted(
    files_2024,
    key=lambda item: item["partition_key"]
):
    partition_key = str(file_record["partition_key"])
    year = int(partition_key[:4])
    month = int(partition_key[5:7])

    summary_path = os.path.join(
        output_dir,
        f"summary_{year}_{month:02d}.csv"
    )

    flows_path = os.path.join(
        output_dir,
        f"flows_{year}_{month:02d}.csv"
    )

    if os.path.exists(summary_path) and os.path.exists(flows_path):
        print(f"{year}-{month:02d}: Already processed")
        continue

    print(f"\nProcessing {year}-{month:02d}", flush=True)

    if month == 1:
        parquet_path = "/content/dewey_january_2024.parquet"

        summary_df, flows_df = extract_monthly_data(
            parquet_path, year, month
        )

    else:
        download_url = get_download_url(file_record)

        if not download_url:
            raise ValueError(
                f"Missing download URL for {partition_key}"
            )

        with tempfile.TemporaryDirectory() as temp_dir:
            parquet_path = os.path.join(
                temp_dir,
                f"dewey_{year}_{month:02d}.parquet"
            )

            with requests.get(
                download_url,
                stream=True,
                timeout=(30, 180)
            ) as response:
                response.raise_for_status()

                with open(parquet_path, "wb") as output_file:
                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            output_file.write(chunk)

            summary_df, flows_df = extract_monthly_data(
                parquet_path, year, month
            )

    if len(summary_df) != 4:
        raise ValueError(
            f"Expected 4 destinations for {year}-{month:02d}, "
            f"found {len(summary_df)}"
        )

    summary_df.to_csv(summary_path, index=False)
    flows_df.to_csv(flows_path, index=False)

    print("Destinations found:", len(summary_df))
    print("Origin flow rows:", len(flows_df))
    print("Saved:", partition_key, flush=True)

    del summary_df, flows_df
    gc.collect()

print("\nALL 2024 MONTHS PROCESSED")

2024-01: Already processed

Processing 2024-02
Destinations found: 4
Origin flow rows: 8000
Saved: 2024-02-01

Processing 2024-03
Destinations found: 4
Origin flow rows: 8000
Saved: 2024-03-01

Processing 2024-04
Destinations found: 4
Origin flow rows: 8000
Saved: 2024-04-01

Processing 2024-05
Destinations found: 4
Origin flow rows: 8000
Saved: 2024-05-01

Processing 2024-06
Destinations found: 4
Origin flow rows: 8000
Saved: 2024-06-01

Processing 2024-07
Destinations found: 4
Origin flow rows: 8000
Saved: 2024-07-01

Processing 2024-08
Destinations found: 4
Origin flow rows: 8000
Saved: 2024-08-01

Processing 2024-09
Destinations found: 4
Origin flow rows: 8000
Saved: 2024-09-01

Processing 2024-10
Destinations found: 4
Origin flow rows: 8000
Saved: 2024-10-01

Processing 2024-11
Destinations found: 4
Origin flow rows: 8000
Saved: 2024-11-01

Processing 2024-12
Destinations found: 4
Origin flow rows: 8000
Saved: 2024-12-01

ALL 2024 MONTHS PROCESSED


In [21]:
import os
import pandas as pd

summary_frames = []
flow_frames = []

for month in range(1, 13):
    summary_path = os.path.join(
        output_dir,
        f"summary_2024_{month:02d}.csv"
    )
    flows_path = os.path.join(
        output_dir,
        f"flows_2024_{month:02d}.csv"
    )

    if not os.path.exists(summary_path):
        raise FileNotFoundError(summary_path)

    if not os.path.exists(flows_path):
        raise FileNotFoundError(flows_path)

    summary_frames.append(pd.read_csv(
        summary_path,
        dtype={"Destination_CBG": str}
    ))

    flow_frames.append(pd.read_csv(
        flows_path,
        dtype={
            "Destination_CBG": str,
            "Origin_CBG": str
        }
    ))

monthly_summary = pd.concat(
    summary_frames,
    ignore_index=True
)

annual_flows = pd.concat(
    flow_frames,
    ignore_index=True
)

monthly_summary = monthly_summary.sort_values(
    ["Mall", "Year", "Month"]
).reset_index(drop=True)

annual_flows = annual_flows.sort_values(
    ["Mall", "Year", "Month", "Origin_CBG"]
).reset_index(drop=True)

summary_output = os.path.join(
    output_dir,
    "Mall_Monthly_Summary.csv"
)

flows_output = os.path.join(
    output_dir,
    "Origin_Destination_Flows.csv"
)

monthly_summary.to_csv(summary_output, index=False)
annual_flows.to_csv(flows_output, index=False)

print("ANNUAL DATA VALIDATION")
print("-" * 50)
print("Monthly summary rows:", len(monthly_summary))
print("Origin-destination flow rows:", len(annual_flows))
print("Unique malls:", monthly_summary["Mall"].nunique())
print("Unique months:", monthly_summary["Month"].nunique())
print("Unique origin CBGs:", annual_flows["Origin_CBG"].nunique())

print("\nMONTHLY ORIGIN RECORDS")
display(
    annual_flows.groupby("Month")
    .size()
    .reset_index(name="Origin_Flow_Rows")
)

print("\nANNUAL MALL SUMMARY")
annual_mall_summary = (
    monthly_summary.groupby("Mall", as_index=False)
    .agg(
        Annual_Device_Counts=("Device_Counts", "sum"),
        Annual_Stop_Counts=("Stop_Counts", "sum"),
        Annual_Reported_Origin_Counts=(
            "Reported_Origin_Counts", "sum"
        )
    )
)

display(annual_mall_summary)

print("\nFILES SAVED")
print(summary_output)
print(flows_output)

ANNUAL DATA VALIDATION
--------------------------------------------------
Monthly summary rows: 48
Origin-destination flow rows: 95803
Unique malls: 4
Unique months: 12
Unique origin CBGs: 16732

MONTHLY ORIGIN RECORDS


,Month,Origin_Flow_Rows
0,1,7803
1,2,8000
2,3,8000
3,4,8000
4,5,8000
5,6,8000
6,7,8000
7,8,8000
8,9,8000
9,10,8000



ANNUAL MALL SUMMARY


,Mall,Annual_Device_Counts,Annual_Stop_Counts,Annual_Reported_Origin_Counts
0,Greenbriar Mall,973928,1861307,629880.0
1,Lenox Square,4072231,13401956,1545519.0
2,Phipps Plaza,2717799,9137661,1242285.0
3,Ponce City Market,2232855,4933694,1014793.0



FILES SAVED
/content/ECON499_Dewey_2024/Mall_Monthly_Summary.csv
/content/ECON499_Dewey_2024/Origin_Destination_Flows.csv


In [22]:
import os
import pandas as pd

summary_frames = []
flow_frames = []

for month in range(1, 13):
    summary_path = os.path.join(
        output_dir,
        f"summary_2024_{month:02d}.csv"
    )
    flows_path = os.path.join(
        output_dir,
        f"flows_2024_{month:02d}.csv"
    )

    if not os.path.exists(summary_path):
        raise FileNotFoundError(summary_path)

    if not os.path.exists(flows_path):
        raise FileNotFoundError(flows_path)

    summary_frames.append(pd.read_csv(
        summary_path,
        dtype={"Destination_CBG": str}
    ))

    flow_frames.append(pd.read_csv(
        flows_path,
        dtype={
            "Destination_CBG": str,
            "Origin_CBG": str
        }
    ))

monthly_summary = pd.concat(
    summary_frames,
    ignore_index=True
)

annual_flows = pd.concat(
    flow_frames,
    ignore_index=True
)

monthly_summary = monthly_summary.sort_values(
    ["Mall", "Year", "Month"]
).reset_index(drop=True)

annual_flows = annual_flows.sort_values(
    ["Mall", "Year", "Month", "Origin_CBG"]
).reset_index(drop=True)

summary_output = os.path.join(
    output_dir,
    "Mall_Monthly_Summary.csv"
)

flows_output = os.path.join(
    output_dir,
    "Origin_Destination_Flows.csv"
)

monthly_summary.to_csv(summary_output, index=False)
annual_flows.to_csv(flows_output, index=False)

print("ANNUAL DATA VALIDATION")
print("-" * 50)
print("Monthly summary rows:", len(monthly_summary))
print("Origin-destination flow rows:", len(annual_flows))
print("Unique malls:", monthly_summary["Mall"].nunique())
print("Unique months:", monthly_summary["Month"].nunique())
print("Unique origin CBGs:", annual_flows["Origin_CBG"].nunique())

print("\nMONTHLY ORIGIN RECORDS")
display(
    annual_flows.groupby("Month")
    .size()
    .reset_index(name="Origin_Flow_Rows")
)

print("\nANNUAL MALL SUMMARY")
annual_mall_summary = (
    monthly_summary.groupby("Mall", as_index=False)
    .agg(
        Annual_Device_Counts=("Device_Counts", "sum"),
        Annual_Stop_Counts=("Stop_Counts", "sum"),
        Annual_Reported_Origin_Counts=(
            "Reported_Origin_Counts", "sum"
        )
    )
)

display(annual_mall_summary)

print("\nFILES SAVED")
print(summary_output)
print(flows_output)

ANNUAL DATA VALIDATION
--------------------------------------------------
Monthly summary rows: 48
Origin-destination flow rows: 95803
Unique malls: 4
Unique months: 12
Unique origin CBGs: 16732

MONTHLY ORIGIN RECORDS


,Month,Origin_Flow_Rows
0,1,7803
1,2,8000
2,3,8000
3,4,8000
4,5,8000
5,6,8000
6,7,8000
7,8,8000
8,9,8000
9,10,8000



ANNUAL MALL SUMMARY


,Mall,Annual_Device_Counts,Annual_Stop_Counts,Annual_Reported_Origin_Counts
0,Greenbriar Mall,973928,1861307,629880.0
1,Lenox Square,4072231,13401956,1545519.0
2,Phipps Plaza,2717799,9137661,1242285.0
3,Ponce City Market,2232855,4933694,1014793.0



FILES SAVED
/content/ECON499_Dewey_2024/Mall_Monthly_Summary.csv
/content/ECON499_Dewey_2024/Origin_Destination_Flows.csv


In [23]:
import os
import zipfile
from google.colab import files

output_dir = "/content/ECON499_Dewey_2024"

zip_path = "/content/ECON499_Dewey_2024_4Malls.zip"

files_to_include = [
    "Mall_Monthly_Summary.csv",
    "Origin_Destination_Flows.csv"
]

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for filename in files_to_include:
        file_path = os.path.join(output_dir, filename)

        if not os.path.exists(file_path):
            raise FileNotFoundError(file_path)

        archive.write(
            file_path,
            arcname=filename
        )

print("EXPORT COMPLETE")
print("-" * 45)
print("ZIP file:", zip_path)
print(
    "ZIP size:",
    round(os.path.getsize(zip_path) / (1024 ** 2), 2),
    "MB"
)

print("\nFILES INCLUDED")
for filename in files_to_include:
    print(filename)

files.download(zip_path)

EXPORT COMPLETE
---------------------------------------------
ZIP file: /content/ECON499_Dewey_2024_4Malls.zip
ZIP size: 0.4 MB

FILES INCLUDED
Mall_Monthly_Summary.csv
Origin_Destination_Flows.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>